# 03 — By-elections and the 2024 national election

**Owner:** C  
**Goal:** post-COVID participation signal per municipality

**Reads:** `data/raw/iec_byelections/, data/raw/iec_npe/`  
**Writes:** `INTERIM / "recent_signals.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## 1. The 2024 national election (NPE) file

Raw file: `data/raw/iec_npe/National.csv`, downloaded from the IEC NPE results page. It holds the
**national ballot** of 29 May 2024: one row per **voting station per party** (52 parties), so each
station's registered voters, spoilt and valid votes are repeated 52 times.

We use the national ballot (not provincial or regional) because every voter gets it, so it is the
cleanest measure of who turned out. Files for other ballots or years (e.g. `827.csv` = 2019) are ignored.

In [ ]:
npe_files = [p for p in (RAW / "iec_npe").glob("*.csv")
             if "national" in p.name.lower() and "provincial" not in p.name.lower()
             and "2019" not in p.name]                                   # 2019 is the baseline (3b)
assert len(npe_files) == 1, f"expected one national-ballot file in {RAW / 'iec_npe'}, found {npe_files}"
NPE_FILE = npe_files[0]

raw = read_any(NPE_FILE)
print(NPE_FILE.name, raw.shape)
print("generated:", raw["Generated_Datetime"].str[:11].unique())
raw.head(3)

## 2. Clean: one row per voting station

* Keep one row per station (`VD_Number`) - the station totals are repeated on every party row.
* Check that each station has a single value for registered / spoilt / valid, and that the party votes add up
  to the valid total.
* Drop **Out of Country** (votes cast at embassies - not a municipality).
* Split `"BUF - Buffalo City"` into `muni_code` and `muni_name`.

In [ ]:
TOTALS = ["Registered_Population", "Spoilt_Votes", "Total_Valid_Votes"]

assert (raw.groupby("VD_Number")[TOTALS].nunique() == 1).all().all(), "a station has conflicting totals"
party_sum = raw.groupby("VD_Number")["Party_Votes"].sum()
stations = raw.drop_duplicates("VD_Number").set_index("VD_Number")
assert (party_sum == stations.loc[party_sum.index, "Total_Valid_Votes"]).all(), "party votes do not add up"

print("stations:", len(stations))
print("dropped (Out of Country):", (stations["Province"] == "Out of Country").sum())
stations = stations[stations["Province"] != "Out of Country"].reset_index()

stations[["muni_code", "muni_name"]] = stations["Municipality"].str.split(" - ", n=1, expand=True)
stations["muni_code"] = stations["muni_code"].str.strip()
stations["muni_name"] = stations["muni_name"].str.strip()
stations = stations.rename(columns={"Province": "province", "Registered_Population": "registered",
                                    "Spoilt_Votes": "spoilt", "Total_Valid_Votes": "valid_votes"})
stations["votes_cast"] = stations["valid_votes"] + stations["spoilt"]

# Some stations have turnout above 100%: people could vote outside their own voting district (section 24A).
# That evens out once we add stations up per municipality.
print("stations with turnout > 1:", (stations["votes_cast"] > stations["registered"]).sum())

## 3. 2024 turnout per municipality

In [ ]:
npe = (stations.groupby(["muni_code"], as_index=False)
               .agg(npe2024_registered=("registered", "sum"),
                    npe2024_votes_cast=("votes_cast", "sum")))
npe["npe2024_turnout"] = npe["npe2024_votes_cast"] / npe["npe2024_registered"]

national_2024 = npe["npe2024_votes_cast"].sum() / npe["npe2024_registered"].sum()
print(f"municipalities: {len(npe)}")
print(f"national turnout 2024 (national ballot, in-country): {national_2024:.2%}   (IEC published, incl. votes abroad: 58.64%)")
print(npe["npe2024_turnout"].describe().round(3).to_string())

## 3b. Pre-COVID baseline: 2019 national election vs 2016 local election

Rural and urban voters always differ between national and local elections, so `08_model` subtracts this
normal gap from the 2024 rebound. 2019 was held on **today's boundaries**, so no voting-district mapping is
needed, and 2016 local → 2019 national mirrors 2021 local → 2024 national (a local election, then a national
one three years later). The rebound then compares like with like: how each municipality's national-election
position changed from 2019 to 2024, minus how its local position changed from 2016 to 2021. COVID only hit the
second pair.

* Raw file: `data/raw/iec_npe/2019 National Results` - one file, or a folder of files (e.g. one per province),
  as CSV or Excel. In Excel files the columns are found by name, so title rows above the table do not matter.
* It must be the **national** ballot: every province then has the same list of parties. The provincial ballot
  has a different list per province and slightly lower turnout (voters outside their home province get only
  the national ballot).
* Some voting-station names contain commas, which shifts the columns in the IEC file, so the station numbers
  are found by position.

In [ ]:
import csv, re
from src.io_utils import _encodings


def read_iec_stations(path):
    """One row per voting station and party from an IEC NPE results CSV."""
    isnum = lambda v: v.strip().isdigit()
    for enc in _encodings(path):
        try:
            with open(path, encoding=enc, newline="") as fh:
                rows = []
                reader = csv.reader(fh)
                next(reader)                                            # header
                for x in reader:
                    k = next((i for i in range(4, len(x) - 2) if all(isnum(v) for v in x[i:i + 3])), None)
                    if k is not None:                                   # registered, spoilt, valid after the name
                        rows.append((x[0], x[1], x[2], int(x[k]), int(x[k + 1]), int(x[k + 2]),
                                     x[k + 3] if len(x) > k + 3 else ""))
            return pd.DataFrame(rows, columns=["province", "municipality", "vd", "registered", "spoilt",
                                               "valid_votes", "party"])
        except UnicodeDecodeError:
            continue
    raise ValueError(f"could not read {path}")


COLUMN_KEYS = {"province": ["province"], "municipality": ["municipal"],
               "vd": ["vdnumber", "votingdistrict", "vdnum"], "registered": ["registered"],
               "spoilt": ["spoilt"], "valid_votes": ["totalvalid", "validvotes"], "party": ["partyname"]}


def turnout_report(t, norm, path, sheet):
    """IEC 'Voter Turnout' report: one row per municipality (registered voters and votes cast, no parties)."""
    votes = next((c for k, c in norm.items() if k.startswith("totalvotescast")), None)
    muni = next((c for k, c in norm.items() if "municipal" in k), None)
    if votes is None or muni is None:
        return None
    reg_keys = ["registered", "completed"] if "completed" in str(votes).lower() else ["registeredpopulation"]
    reg = next((c for k, c in norm.items() if all(key in k for key in reg_keys)), None)         or next((c for k, c in norm.items() if "registered" in k), None)
    print(f"  {path.name} / {sheet}: municipality turnout report ({reg!r}, {votes!r})")
    return pd.DataFrame({"province": path.stem, "municipality": t[muni].astype(str), "vd": t[muni].astype(str),
                         "registered": t[reg], "spoilt": 0, "valid_votes": t[votes], "party": "",
                         "ballot": sheet})


def read_iec_excel(path):
    """The same columns as read_iec_stations, from an IEC Excel download: station results, or the
    municipality turnout report (every sheet that has one of the two)."""
    parts = []
    for sheet in pd.ExcelFile(path).sheet_names:
        t = read_any(path, sheet_name=sheet)
        norm = {str(c).lower().replace(" ", "").replace("_", ""): c for c in t.columns}
        found = {name: next((c for k, c in norm.items() if any(key in k for key in keys)), None)
                 for name, keys in COLUMN_KEYS.items()}
        if any(v is None for k, v in found.items() if k != "party"):
            part = turnout_report(t, norm, path, sheet)
            if part is None:
                print(f"  {path.name} / {sheet}: no results table, skipped (columns: {list(t.columns)[:8]})")
                continue
            for c in ["registered", "valid_votes"]:
                part[c] = pd.to_numeric(part[c], errors="coerce")
            parts.append(part.dropna(subset=["registered", "valid_votes"]))
            continue
        part = pd.DataFrame({k: t[v] for k, v in found.items() if v is not None})
        part = part.reindex(columns=list(COLUMN_KEYS))
        part["ballot"] = sheet
        for c in ["vd", "registered", "spoilt", "valid_votes"]:
            part[c] = pd.to_numeric(part[c], errors="coerce")
        part = part.dropna(subset=["vd", "registered"])                  # totals and blank rows
        part["vd"] = part["vd"].astype("int64").astype(str)
        parts.append(part)
    assert parts, f"{path.name}: no sheet with station results"
    return pd.concat(parts)


found = list((RAW / "iec_npe").glob("*2019*"))
files19 = [f for p in found for f in (sorted(p.rglob("*")) if p.is_dir() else [p])
           if f.suffix.lower() in (".csv", ".xlsx", ".xls") and not f.name.startswith("~$")
           and not re.search(r" \(\d+\)\.csv$", f.name)]
print("2019 files:", [f.name for f in files19] or "none")

if not files19:
    print("No 2019 national results in", RAW / "iec_npe", "- 2019 baseline columns will be empty.")
    npe19 = pd.DataFrame(columns=["muni_code", "npe2019_turnout", "npe2019_registered", "npe2019_votes_cast"])
else:
    raw19 = pd.concat(read_iec_stations(f) if f.suffix.lower() == ".csv" else read_iec_excel(f) for f in files19)
    raw19["vd"] = raw19["vd"].astype(str).str.strip()
    raw19 = raw19[~raw19["province"].str.startswith("Out of Country")]

    parties = raw19.groupby("province")["party"].nunique()
    if parties.nunique() > 1:
        print("WARNING: the party list differs by province - this looks like the PROVINCIAL ballot, "
              "not the national one:\n", parties.to_string())
    if "ballot" in raw19 and raw19["ballot"].fillna("").str.contains("provincial", case=False).any():
        print("NOTE: these are PROVINCIAL-ballot figures (sheets:", sorted(raw19["ballot"].dropna().unique()),
              ") - fine for the baseline, which compares each municipality with the country, "
              "but national turnout will be a little below the IEC's 66.05% (national ballot).")

    st19 = raw19.drop_duplicates("vd").copy()
    st19["muni_code"] = st19["municipality"].str.split(" - ", n=1).str[0].str.strip()
    is_code = st19["muni_code"].str.fullmatch(r"[A-Z]{3}|[A-Z]{2,3}\d{3}")     # e.g. BUF, EC101, KZN226
    if (~is_code).any():
        print("  rows without a municipality code, dropped (totals etc.):",
              st19.loc[~is_code, "municipality"].unique()[:10].tolist())
    st19 = st19[is_code]
    npe19 = (st19.groupby("muni_code", as_index=False)
                 .agg(npe2019_registered=("registered", "sum"), votes_ok=("valid_votes", "sum"),
                      spoilt=("spoilt", "sum")))
    npe19["npe2019_votes_cast"] = npe19.pop("votes_ok") + npe19.pop("spoilt")
    npe19["npe2019_turnout"] = npe19["npe2019_votes_cast"] / npe19["npe2019_registered"]

    national_2019 = npe19["npe2019_votes_cast"].sum() / npe19["npe2019_registered"].sum()
    print(f"stations: {len(st19)}   municipalities: {len(npe19)}")
    print(f"national turnout 2019: {national_2019:.2%}   (IEC published, national ballot: 66.05%)")
    assert len(npe19) == 213, f"2019 should have the same 213 municipalities as today; got {len(npe19)}, e.g. {raw19['municipality'].unique()[:5].tolist()}"

## 4. By-elections since November 2021

Owner C: put the by-election result files in `data/raw/iec_byelections/` and fill in the cell below so it
produces one row per municipality with `byelection_turnout_mean` and `byelection_count`.

Until those files are there, both columns stay **empty** (not 0): 0 would claim that a municipality had no
by-elections, which we do not know yet.

In [ ]:
be_files = inventory(RAW / "iec_byelections")
if len(be_files) == 0:
    print("No by-election files yet - byelection columns will be empty.")
    be_agg = pd.DataFrame(columns=["muni_code", "byelection_turnout_mean", "byelection_count"])
else:
    display(be_files.head(20))
    # be = ...  read and clean the files, one row per by-election ward
    # be["turnout"] = be["votes_cast"] / be["registered"]
    # be_agg = be.groupby("muni_code").agg(byelection_turnout_mean=("turnout", "mean"),
    #                                      byelection_count=("turnout", "size")).reset_index()
    raise NotImplementedError("by-election files found - finish this cell")

## 5. Merge and check

One row per municipality, 213 rows, same codes as the results file. When the by-election data is in,
municipalities with no by-election get `byelection_count = 0` - a zero is information, not a missing value.

In [ ]:
out = (npe[["muni_code", "npe2024_turnout", "npe2024_registered", "npe2024_votes_cast"]]
         .merge(npe19, on="muni_code", how="left")
         .merge(be_agg, on="muni_code", how="left"))
if len(be_agg):
    out["byelection_count"] = out["byelection_count"].fillna(0).astype(int)

out = out[["muni_code", "byelection_turnout_mean", "byelection_count",
           "npe2024_turnout", "npe2024_registered", "npe2024_votes_cast",
           "npe2019_turnout", "npe2019_registered", "npe2019_votes_cast"]].sort_values("muni_code")

assert len(out) == 213 and out["muni_code"].is_unique, "expected 213 unique municipalities"
assert not out["muni_code"].str.startswith("DC").any(), "district municipalities must not be rows"
assert out["npe2024_turnout"].between(0, 1).all(), "turnout must be a fraction"

results_file = INTERIM / "results_lge_2011_2021.csv"
if results_file.exists():
    codes_2021 = set(pd.read_csv(results_file).query("election == 2021")["muni_code"])
    assert set(out["muni_code"]) == codes_2021, set(out["muni_code"]) ^ codes_2021
    print("codes match the 2021 results file")

## Save output

In [ ]:
out.to_csv(INTERIM / "recent_signals.csv", index=False)
print(out.shape)
out.head()

## Columns in `recent_signals.csv`

| Column | Meaning |
|---|---|
| `muni_code` | Municipality code - join on this |
| `byelection_turnout_mean` | Mean turnout in by-elections since Nov 2021 (empty until the by-election data is in) |
| `byelection_count` | Number of by-elections since Nov 2021, 0 where there were none (empty until then) |
| `npe2024_turnout` | Turnout in the 29 May 2024 national election, national ballot, as a fraction |
| `npe2024_registered`, `npe2024_votes_cast` | The counts behind it, so national totals can be recomputed |
| `npe2019_turnout`, `npe2019_registered`, `npe2019_votes_cast` | The same for the 2019 national election (the pre-COVID baseline `08_model` uses) |